In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv(override=True)
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

model_name = os.getenv("MODEL", "openai/gpt-6-luna")

# **Custom Prompt Templates in LangChain**

LangChain is a Python library for building apps that use an LLM (large language model), the kind of AI that writes text from a prompt. The ideas below are about how you build those prompts.

- **Tailor-Made**: Craft prompts that fit your model like a glove.

- **Full Control**: Dictate every detail from instructions to formatting.

- **Dynamic Inputs**: Plug in specific data as needed for each task.

A **prompt template** is a reusable recipe for a prompt, with blanks (called *input variables*) that you fill in at run time. LangChain's `PromptTemplate` is the built-in version.

A **custom template** is a class you write yourself. You control how it builds the prompt in its `format()` method, as shown in the `FunctionExplainerPromptTemplate` cell below.

Custom templates are like a bespoke suit, made to fit your model's unique requirements.


# **Custom vs. Default Prompt Templates**

- **Default Templates**: Like using a template for a letter, where you just fill in the blanks. In this notebook, this is `PromptTemplate.from_template`, in the code cell after the Note.

- **Custom Templates**: Like writing a script for a play, where you have the freedom to craft the entire scene. In this notebook, this is the `FunctionExplainerPromptTemplate` class.


> **Note:** You may see a `UserWarning` about Pydantic V1 when running on Python 3.14 or later. It comes from LangChain's internal compatibility code. It does not change the results in this notebook.


In [2]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("Hello {name}!")

prompt = template.format(name="Mohammed")

print(prompt)

/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Hello Mohammed!


**Custom templates let you:**

- **Programmatically Craft Prompts**: Build prompts on the fly, tailored to the task at hand.

- **Example - `FunctionExplainerPromptTemplate`**: this template takes a Python function and asks the LLM to explain it in plain English. Here is what happens, step by step:

  1. **Function input**: you pass the function object itself, not its name as text. The input is named `function_name`, but it holds a function. That is why the code reads `.__name__` to get the name. The name is a little misleading, as the model's reply in the stored output also points out.
  2. **Code inspection**: `inspect` is Python's built-in module for examining code. `inspect.getsource()` returns the function's source code.
  3. **Prompt assembly**: `format()` fills in the `PROMPT` text with the name and source code, and returns a plain string.
  4. **Model call**: the string is sent to the LLM, which returns an explanation. The `llm.invoke` cell and the chain cell both do this.

  Flow: `Function object → format() → Prompt string → LLM → plain text`

This approach is like having a Swiss Army knife for prompt creation, giving you the tools to construct exactly what you need.


### How the custom class is built

The `FunctionExplainerPromptTemplate` cell defines three things:

- **`PROMPT`**: a text template with `{function_name}` and `{source_code}` blanks. It works the same way as the default template (`PromptTemplate.from_template`).
- **`FunctionExplainerPromptTemplate`**: a class that inherits from LangChain's `StringPromptTemplate` base class. Inheriting from it lets the template be used in a chain with the `|` operator, as in the chain cell.
- **`field_validator`**: a check on the `input_variables` field. It makes sure the template accepts exactly one input variable, called `function_name`. It uses Pydantic, a Python library for validating data.


In [3]:
import inspect
from langchain_core.prompts import StringPromptTemplate
from pydantic import BaseModel, field_validator

def get_source_code(function_name):
    """Return the source code of the provided function."""
    # Using the inspect module to get the source code of the function
    return inspect.getsource(function_name)

# Template string for the prompt that will be sent to the language model
PROMPT = """Given the function name and source code, generate an English language explanation of the function.
Function Name: {function_name}

Source Code:
{source_code}

Explanation:
"""

class FunctionExplainerPromptTemplate(StringPromptTemplate, BaseModel):
    """A custom prompt template that takes in the function name as input and formats the prompt template to provide the source code of the function."""

    @field_validator("input_variables")
    def validate_input_variables(cls, v):
        """Validate that the input variables are correct."""
        # Ensuring that the only input variable is 'function_name'
        if len(v) != 1 or "function_name" not in v:
            raise ValueError("function_name must be the only input_variable.")
        return v

    def format(self, **kwargs) -> str:
        """Format the prompt using the function's name and source code."""
        # Retrieve the source code of the provided function
        source_code = get_source_code(kwargs["function_name"])

        # Format the PROMPT string using the function name and its source code
        prompt = PROMPT.format(
            function_name=kwargs["function_name"].__name__,
            source_code=source_code
        )
        return prompt

    def _prompt_type(self) -> str:
        """Return the type of prompt."""
        return "function-explainer"

In [4]:
fn_explainer = FunctionExplainerPromptTemplate(input_variables=["function_name"])


In [5]:
type(fn_explainer)

__main__.FunctionExplainerPromptTemplate

In [6]:
# Generate a prompt for the function "get_source_code"
prompt = fn_explainer.format(function_name=get_source_code)

print(prompt)

Given the function name and source code, generate an English language explanation of the function.
Function Name: get_source_code

Source Code:
def get_source_code(function_name):
    """Return the source code of the provided function."""
    # Using the inspect module to get the source code of the function
    return inspect.getsource(function_name)


Explanation:



### Sending the prompt to the model

The output of `format()` is just a string. The `llm.invoke` cell sends that string to the model, and the reply is in `response.content`.

The chain cell joins the steps into one chain with `|`. The template's output feeds into the model, and `StrOutputParser()` turns the model's reply into plain text.


In [7]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(model=model_name)

response = llm.invoke(prompt)

response.content

'Returns the source code for the provided function using Python’s `inspect.getsource`. Despite the parameter name, `function_name` should be a function object, not a name string.'

In [8]:
from langchain_core.output_parsers import StrOutputParser

llm_chain = fn_explainer | llm | StrOutputParser()

for chunk in llm_chain.stream({"function_name":get_source_code}):
  print(chunk, end="", flush=True)

`get_source_code` returns the source code for the supplied function by calling `inspect.getsource` on it. The argument should be a function or other object supported by `inspect.getsource`, rather than just the function’s name as a string.

# Another example: `AlgorithmOptimizerPromptTemplate`

This template follows the same pattern as the one above, with three differences:

- **Input name**: `algorithm_function` instead of `function_name`.
- **Instruction**: it asks the model to suggest optimizations and return improved code, instead of explaining the function.
- **Validator**: the `field_validator` checks that `algorithm_function` is the only input variable.

We will use `recursive_factorial` as the example algorithm (defined in the `recursive_factorial` cell below). It calculates a factorial by calling itself.


In [9]:
# Template string for the algorithm optimization prompt
ALGO_PROMPT = """Given the algorithm below, suggest optimizations or potential \
improvements, and return the optimized code.

Algorithm Name: {algorithm_name}

Source Code:
{source_code}

Suggestions:
"""

class AlgorithmOptimizerPromptTemplate(StringPromptTemplate, BaseModel):
    """A custom prompt template that takes an algorithm as input and formats the prompt template to request optimization suggestions."""

    @field_validator("input_variables")
    def validate_input_variables(cls, v):
        """Validate that the input variables are correct."""
        if len(v) != 1 or "algorithm_function" not in v:
            raise ValueError("algorithm_function must be the only input_variable.")
        return v

    def format(self, **kwargs) -> str:
        """Format the prompt using the algorithm's name and source code."""

        # Retrieve the source code of the provided algorithm
        source_code = get_source_code(kwargs["algorithm_function"])

        # Format the ALGO_PROMPT string using the algorithm name and its source code
        prompt = ALGO_PROMPT.format(
            algorithm_name=kwargs["algorithm_function"].__name__,
            source_code=source_code
        )
        return prompt

    def _prompt_type(self) -> str:
        """Return the type of prompt."""
        return "algorithm-optimizer"


In [10]:
def recursive_factorial(n: int) -> int:
    """Calculate factorial of a number using recursion."""
    if n == 0:
        return 1
    else:
        return n * recursive_factorial(n-1)


In [11]:
# Instantiate the AlgorithmOptimizerPromptTemplate with the appropriate input variable
algo_optimizer = AlgorithmOptimizerPromptTemplate(input_variables=["algorithm_function"])

# Generate a prompt for the function "recursive_factorial"
prompt = algo_optimizer.format(algorithm_function=recursive_factorial)

print(prompt)

Given the algorithm below, suggest optimizations or potential improvements, and return the optimized code.

Algorithm Name: recursive_factorial

Source Code:
def recursive_factorial(n: int) -> int:
    """Calculate factorial of a number using recursion."""
    if n == 0:
        return 1
    else:
        return n * recursive_factorial(n-1)


Suggestions:



In [12]:
result = llm.invoke(prompt)

print(result.content)

The recursive version adds a Python function call for every factor and can exceed Python’s recursion limit for large inputs. Use the standard library’s optimized implementation instead; it also raises `ValueError` for negative inputs and `TypeError` for non-integer inputs.

```python
import math

def factorial(n: int) -> int:
    """Return n! for a non-negative integer n."""
    return math.factorial(n)
```

If callers rely on the original function name, keep it as a compatibility wrapper:

```python
def recursive_factorial(n: int) -> int:
    """Return n! (legacy function name; calculation is not recursive)."""
    return math.factorial(n)
```

This avoids recursion overhead and stack limits. For a manual implementation, an iterative loop is a good alternative to recursion.


### Streaming the answer

The chain in the streaming cell below uses `.stream()` instead of `.invoke()`. The model's answer arrives in small pieces (chunks), and each piece is printed as it arrives instead of all at once. Streaming means the first words can appear before the whole reply is ready.


In [13]:
llm_chain = algo_optimizer | llm | StrOutputParser()

for chunk in llm_chain.stream({"algorithm_function":recursive_factorial}):
  print(chunk, end="", flush=True)

The recursive version makes one function call per factor, so large inputs can exceed Python’s recursion limit. Python’s `math.factorial` avoids that overhead and uses an optimized implementation.

```python
from math import factorial

def recursive_factorial(n: int) -> int:
    """Return n!, using Python's optimized factorial implementation."""
    return factorial(n)
```

The function name is retained for compatibility, but it no longer uses recursion. If you can change callers, consider renaming it to `factorial` or `calculate_factorial`. `math.factorial` also raises `ValueError` for negative inputs and `TypeError` for non-integer inputs.

### Think critically

- **Results vary.** The `llm.invoke` cell and the chain cell send the same prompt, but the replies differ in wording and detail. Your results will differ too.
- **Check the suggestions.** The model suggests `math.factorial` in both the invoke output and the streamed output. This notebook does not run that suggestion. Test any model-generated code before you trust it.
- **Test the claims too.** The model says `math.factorial` raises `ValueError` for negative inputs and `TypeError` for non-integer inputs. These are the model's claims. You can check them with a quick call such as `math.factorial(-1)`, which this notebook does not run.
- **Protect private code.** The code is sent to an external model provider (OpenRouter, in this notebook). Do not send confidential code, private data or API keys.
- **Know the limits of `inspect`.** `inspect.getsource()` cannot read the source of built-in functions such as `len`. It raises a `TypeError` instead.
- **Recursion has a cost.** `recursive_factorial(-1)` never reaches the base case `n == 0`. It keeps calling itself until Python stops it with a `RecursionError`.
